# Student 1 — Simple Python Code (V1 and V2)
**Problem 14: Triangle Soup vs. Indexed Mesh**


what implemented was the first two versions. V1 stores triangles separately and checks them one by one. V2 stores shared vertices once and uses the Möller–Trumbore intersection formula. I compare their answers and speed.”

### Cell 1 — Import and create a triangle

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

# Three vertices and one triangular face
vertices = np.array([[0.,0.,0.], [4.,0.,0.], [0.,4.,0.]])
faces = np.array([[0,1,2]])

origin = np.array([1.,1.,3.])
direction = np.array([0.,0.,-1.])

### Cell 2 — V1: Triangle Soup (plane + edge tests)
**Explain:** “V1 stores all three points for each triangle. It finds where the ray meets the triangle’s plane, then checks whether that point is inside the triangle.”

In [ ]:
def v1(triangles, origin, direction):
    best_id, best_t = -1, float("inf")
    for i, (a,b,c) in enumerate(triangles):
        normal = np.cross(b-a, c-a)
        denom = np.dot(normal, direction)
        if abs(denom) < 1e-9: continue  # parallel ray
        t = np.dot(normal, a-origin) / denom
        if t <= 0: continue
        point = origin + t*direction
        inside = all(
            np.dot(normal, np.cross(q-p, point-p)) >= -1e-9
            for p,q in ((a,b),(b,c),(c,a))
        )
        if inside and t < best_t:
            best_id, best_t = i, t
    return best_id, best_t

### Cell 3 — V2: Indexed Mesh (Möller–Trumbore)
**Explain:** “V2 stores each vertex only once. A face contains three indices pointing to vertices. Möller–Trumbore calculates whether the ray intersects the triangle using cross and dot products.”

In [ ]:
def v2(vertices, faces, origin, direction):
    best_id, best_t = -1, float("inf")
    for i, (i0,i1,i2) in enumerate(faces):
        a,b,c = vertices[[i0,i1,i2]]
        e1, e2 = b-a, c-a
        p = np.cross(direction, e2)
        det = np.dot(e1, p)
        if abs(det) < 1e-9: continue
        inv = 1/det
        T = origin-a
        u = np.dot(T,p)*inv
        if u < 0 or u > 1: continue
        q = np.cross(T,e1)
        v = np.dot(direction,q)*inv
        if v < 0 or u+v > 1: continue
        t = np.dot(e2,q)*inv
        if 0 < t < best_t:
            best_id, best_t = i,t
    return best_id,best_t

### Cell 4 — Test that both versions give the same answer
**Explain:** “The ray should hit triangle 0 at distance 3. Both algorithms must agree.”

In [ ]:
answer1 = v1(vertices[faces], origin, direction)
answer2 = v2(vertices, faces, origin, direction)
print("V1:", answer1)
print("V2:", answer2)
assert answer1[0] == answer2[0] == 0
assert abs(answer1[1]-3) < 1e-6
assert abs(answer2[1]-3) < 1e-6
print("PASS: both versions agree!")

### Cell 5 — Simple speed comparison and Q1 memory
**Explain:** “I create a grid of triangles, send rays through it, and measure how long each algorithm takes. Both still check every triangle. V2 saves memory, but is not automatically faster.”

In [ ]:
# Grid with 200 triangles (10 x 10 squares, 2 triangles each)
side = 10
points = np.array([[x,y,0.] for y in range(side+1) for x in range(side+1)],float)
tri_faces = []
for y in range(side):
    for x in range(side):
        a = y*(side+1)+x
        tri_faces += [[a,a+1,a+side+1],[a+1,a+side+2,a+side+1]]
tri_faces = np.array(tri_faces)
soup = points[tri_faces]

rays = [np.array([x,y,2.]) for x in np.linspace(0.5,9.5,5)
                            for y in np.linspace(0.5,9.5,5)]
times = []
for label, algorithm in [
    ("V1", lambda o: v1(soup,o,np.array([0.,0.,-1.]))),
    ("V2", lambda o: v2(points,tri_faces,o,np.array([0.,0.,-1.])))
]:
    start = time.perf_counter()
    answers = [algorithm(o) for o in rays]
    duration = time.perf_counter()-start
    times.append(duration)
    print(label, "seconds:", round(duration,4))

plt.bar(["V1","V2"], times)
plt.ylabel("Time for 25 rays (seconds)")
plt.title("Simple V1 vs V2 speed test")
plt.show()

# Q1 uses the professor's 50,000 vertices and 100,000 faces
V,F = 50000,100000
print("Triangle Soup:", 9*F*4/1e6, "MB")
print("Indexed Mesh:", (3*V+3*F)*4/1e6, "MB")
print("Memory reduction: 50%")


1. **Ray:** `origin + t × direction`.
2. **V1:** Stores 3 full vertices per triangle; checks each triangle using plane and edge tests.
3. **V2:** Stores unique vertices and face indices; uses Möller–Trumbore.
4. **Correctness:** Both return triangle ID `0`, distance `3` for the example.
5. **Complexity:** Both are **O(R × N)**. V2 saves vertex memory; it does **not** eliminate brute-force triangle checks.

